# 飞书 (Feishu/Lark) API 全流程验证

> **目标**：验证飞书 Open API 的完整能力链路
> **覆盖**：认证 → 创建 → 追加(文本/标题/列表/代码/公式/表格/图片) → 读取 → 块操作 → 搜索 → 清理
> **认证**：App ID + App Secret → tenant_access_token

In [32]:
# 创建测试文档（通用双 Token 策略）

# 模式 A: use_user_token=True  → user token 创建，用户是拥有者，可直接迁入 Wiki
# 模式 B: use_user_token=False → tenant token 创建，自动分享给当前 user token 用户

title = f"API验证 - {datetime.now().strftime('%H:%M:%S')}"

# ===== 配置 =====
# True:  user token 创建 → 用户是拥有者 → 可迁入 Wiki
# False: tenant token 创建 → 自动分享给 user token 用户 → 双方都能编辑
USE_USER_TOKEN = False
# ================

result = client.api("POST", "/docx/v1/documents", json_data={"title": title}, use_user_token=USE_USER_TOKEN)
document_id = result["document"]["document_id"]

print(f"[OK] Created doc: {title}")
print(f"     document_id: {document_id}")
print(f"     URL: https://feishu.cn/docx/{document_id}")

if USE_USER_TOKEN:
    print(f"     模式: user token 创建（用户是拥有者，可迁入 Wiki）")
else:
    print(f"     模式: tenant token 创建（已自动分享给当前 user token 用户）")
    print(f"     提示: 如需分享给其他人，见下方 1.5 节")

[OK] FeishuClient initialized
       Token valid: True
       Expire in: 7095s


## 1. 创建文档

验证 `POST /docx/v1/documents` 创建 docx 格式文档。

In [33]:
import time
import json

# 权限分享 —— 分享给【额外用户】
#
# 创建文档时已自动将 full_access 权限分享给当前 user_access_token 用户。
# 这里仅用于分享给其他人（同事、上级等），无需操作则留空。

TEST_MOBILE = ""   # 额外用户的手机号
TEST_EMAIL  = ""   # 额外用户的企业邮箱

MY_OPEN_ID = None

if TEST_MOBILE or TEST_EMAIL:
    try:
        payload = {}
        if TEST_EMAIL:  payload['emails']  = [TEST_EMAIL]
        if TEST_MOBILE: payload['mobiles'] = [TEST_MOBILE]
        
        print("--- 正在查询用户 open_id... ---")
        ident_res = client.api(
            'POST', 
            '/contact/v3/users/batch_get_id',
            json_data=payload,
            params={'user_id_type': 'open_id'}
        )
        
        user_list = ident_res.get('user_list', [])
        if user_list:
            MY_OPEN_ID = user_list[0].get('user_id')
            print(f"✅ 查到 open_id: {MY_OPEN_ID}")
        else:
            print("❌ 未找到用户")
    except Exception as e:
        print(f"⚠️ 查询失败: {e}")

    if MY_OPEN_ID:
        time.sleep(1.5)
        try:
            client.api(
                'POST', 
                f'/drive/v1/permissions/{document_id}/members',
                params={'type': 'docx'}, 
                json_data={
                    'member_type': 'openid',
                    'member_id': MY_OPEN_ID,
                    'perm': 'full_access'
                }
            )
            print(f"✅ 权限已下放给 [{MY_OPEN_ID}] (Full Access)")
        except Exception as e:
            print(f"⚠️ 权限下放失败: {e}")
else:
    print("ℹ️ 未指定额外用户，跳过手动分享")
    print("   （当前 user token 用户已在创建时自动获得 full_access）")

[OK] Created doc: API验证 - 01:22:55
     document_id: HgGBdPSVqocd5UxYCoocxh7cnYc
     URL: https://feishu.cn/docx/HgGBdPSVqocd5UxYCoocxh7cnYc


## 1.5 文档权限分享

飞书应用创建的文档默认只有应用自己能编辑。为了让指定用户能在飞书客户端里查看和编辑这篇文档，需要在创建后立刻把权限分享出去。

流程：
1. 通过 `POST /contact/v3/users/batch_get_id` 用邮箱/手机号/用户ID 换取用户的 `open_id`
2. 通过 `POST /drive/v1/permissions/{document_id}/members` 把文档分享给该用户

> **注意**：应用需要在后台申请 `drive:drive` 权限，且被查询的用户必须在当前企业租户内。

In [34]:
import time
import json

# ================== 只改这一行 ==================
TEST_MOBILE = "13767824826"   # 填你的手机号
TEST_EMAIL  = ""              # 或填你的企业邮箱
# ==============================================

MY_OPEN_ID = None

# Step 1: 自动查询用户 open_id
try:
    payload = {}
    if TEST_EMAIL:  payload['emails']  = [TEST_EMAIL]
    if TEST_MOBILE: payload['mobiles'] = [TEST_MOBILE]
    
    if payload:
        print("--- 正在查询用户 open_id... ---")
        ident_res = client.api(
            'POST', 
            '/contact/v3/users/batch_get_id',
            json_data=payload,
            params={'user_id_type': 'open_id'}
        )
        
        # 自动解析返回值，不用手动复制粘贴
        user_list = ident_res.get('user_list', [])
        if user_list:
            # 飞书返回 open_id 时，字段名叫做 user_id
            MY_OPEN_ID = user_list[0].get('user_id')
            print(f"✅ 查到 open_id: {MY_OPEN_ID}")
        else:
            print("❌ 未找到用户，请确认手机号/邮箱正确，且应用有通讯录权限")
    else:
        print("ℹ️ 请先在代码里填上 TEST_MOBILE 或 TEST_EMAIL")
except Exception as e:
    print(f"⚠️ 查询失败: {e}")

# Step 2: 自动分享权限（如果查到了 ID）
if MY_OPEN_ID:
    # 文档刚创建，Drive 索引可能有延迟，稍等一下
    time.sleep(1.5)
    
    try:
        client.api(
            'POST', 
            f'/drive/v1/permissions/{document_id}/members',
            params={'type': 'docx'}, 
            json_data={
                'member_type': 'openid',
                'member_id': MY_OPEN_ID,
                'perm': 'full_access'
            }
        )
        print(f"✅ 权限已成功下放给 [{MY_OPEN_ID}] (Full Access)")
    except Exception as e:
        print(f"⚠️ 权限下放失败: {e}")
else:
    print("ℹ️ 未获取到用户 ID，跳过权限分享")

--- 正在查询用户 open_id... ---
✅ 查到 open_id: None
ℹ️ 未获取到用户 ID，跳过权限分享


## 1.6 取消文档权限

验证 `DELETE /drive/v1/permissions/{token}/members/{member_id}` 移除协作者权限。

> 把刚才 1.5 分享出去的权限收回来。

In [35]:
# # 取消文档权限（移除协作者）
# if 'MY_OPEN_ID' in locals() and MY_OPEN_ID:
#     try:
#         client.api(
#             "DELETE",
#             f"/drive/v1/permissions/{document_id}/members/{MY_OPEN_ID}",
#             params={"type": "docx", "member_type": "openid"}
#         )
#         print(f"✅ 已取消用户 [{MY_OPEN_ID}] 的文档权限")
#     except Exception as e:
#         print(f"⚠️ 取消权限失败: {e}")
# else:
#     print("ℹ️ 未获取到用户 ID，跳过取消权限")

## 2. 追加文本内容（md_to_blocks 方式）

使用 `md_to_blocks()` 将 Markdown 文本转换为飞书 Block 格式，然后追加到文档末尾。

In [36]:
content = """## 文本追加测试

这是一段普通文本，用于验证 feishu_doc_append 的 content 模式。

- 支持无序列表
- 自动转换为飞书 bullet block

1. 有序列表项1
2. 有序列表项2

> 引用块测试

段落之间需要空行分隔。"""

blocks = md_to_blocks(content)
print(f"[INFO] Converted to {len(blocks)} blocks")
for b in blocks:
    print(f"  {block_type_name(b['block_type'])}: {extract_text_from_block(b)[:30]}...")

result = client.api(
    "POST",
    f"/docx/v1/documents/{document_id}/blocks/{document_id}/children",
    json_data={"children": blocks}
)
print(f"[OK] Appended {len(blocks)} blocks")

[INFO] Converted to 8 blocks
  heading2: 文本追加测试...
  text: 这是一段普通文本，用于验证 feishu_doc_appen...
  bullet: 支持无序列表...
  bullet: 自动转换为飞书 bullet block...
  ordered: 有序列表项1...
  ordered: 有序列表项2...
  quote: 引用块测试...
  text: 段落之间需要空行分隔。...
[OK] Appended 8 blocks


## 3. 追加代码块

验证 `block_type: 14` code block 的创建。

In [37]:
code_blocks = [
    make_heading_block("代码块测试", level=2),
    make_text_block("Python 示例代码:"),
    make_code_block("def hello():\n    print('Hello, Feishu!')\n\nhello()"),
]
result = client.api(
    "POST",
    f"/docx/v1/documents/{document_id}/blocks/{document_id}/children",
    json_data={"children": code_blocks}
)
print("[OK] Appended code block")

[OK] Appended code block


## 4. 追加数学公式

飞书原生支持公式，通过 `text_element_style.formula` 或 `inline_formula` 设置。

**注意**：公式内容必须是 KaTeX 语法，`\frac` 等命令需要双反斜杠 `\\frac`。

In [38]:
# 追加数学公式（飞书用 equation 元素，不是 text_run.formula）
formula_blocks = [
    make_heading_block("数学公式测试", level=2),
    make_text_block("一元二次方程求根公式："),
    {
        "block_type": 2,
        "text": {
            "elements": [
                {
                    "equation": {
                        "content": "x = \\frac{-b \\pm \\sqrt{b^2-4ac}}{2a}"
                    }
                }
            ]
        }
    },
]
result = client.api(
    "POST",
    f"/docx/v1/documents/{document_id}/blocks/{document_id}/children",
    json_data={"children": formula_blocks}
)
print("[OK] Appended formula blocks")

[OK] Appended formula blocks


### 4.1 GRPO Reward 与 KL 散度

GRPO（Group Relative Policy Optimization）的核心目标函数，包含策略比率裁剪、组内相对优势估计和 KL 散度惩罚项。

In [39]:
grpo_blocks = [
    make_heading_block("GRPO 目标函数", level=3),
    make_text_block("策略比率与裁剪："),
    {
        "block_type": 2,
        "text": {
            "elements": [{
                "equation": {
                    "content": "r_i(\\theta) = \\frac{\\pi_{\\theta}(o_i|q)}{\\pi_{\\theta_{old}}(o_i|q)}"
                }
            }]
        }
    },
    make_text_block("组内相对优势（组大小 G）："),
    {
        "block_type": 2,
        "text": {
            "elements": [{
                "equation": {
                    "content": "\\hat{A}_i = \\frac{r_i - \\text{mean}(\\{r_j\\}_{j=1}^{G})}{\\text{std}(\\{r_j\\}_{j=1}^{G})}"
                }
            }]
        }
    },
    make_text_block("带 KL 惩罚的 GRPO 目标："),
    {
        "block_type": 2,
        "text": {
            "elements": [{
                "equation": {
                    "content": "J_{GRPO}(\\theta) = \\mathbb{E}_{q,\\{o_i\\}}\\left[ \\frac{1}{G}\\sum_{i=1}^{G}\\left( \\min\\left( r_i\\hat{A}_i, \\text{clip}(r_i, 1-\\epsilon, 1+\\epsilon)\\hat{A}_i \\right) - \\beta D_{KL}(\\pi_{\\theta} \\| \\pi_{ref}) \\right) \\right]"
                }
            }]
        }
    },
]

client.api(
    "POST",
    f"/docx/v1/documents/{document_id}/blocks/{document_id}/children",
    json_data={"children": grpo_blocks}
)
print("[OK] Appended GRPO formulas")

[OK] Appended GRPO formulas


### 4.2 偏微分方程（热方程）

经典的热传导方程，包含拉普拉斯算子与扩散系数。

In [40]:
pde_blocks = [
    make_heading_block("热传导方程", level=3),
    make_text_block("三维热方程："),
    {
        "block_type": 2,
        "text": {
            "elements": [{
                "equation": {
                    "content": "\\frac{\\partial u}{\\partial t} = \\alpha \\nabla^2 u = \\alpha \\left( \\frac{\\partial^2 u}{\\partial x^2} + \\frac{\\partial^2 u}{\\partial y^2} + \\frac{\\partial^2 u}{\\partial z^2} \\right)"
                }
            }]
        }
    },
    make_text_block("初始条件与边界条件："),
    {
        "block_type": 2,
        "text": {
            "elements": [{
                "equation": {
                    "content": "u(x,y,z,0) = \\varphi(x,y,z), \\quad u|_{\\partial \\Omega} = g(x,y,z,t)"
                }
            }]
        }
    },
    make_text_block("分离变量法得到的解析解："),
    {
        "block_type": 2,
        "text": {
            "elements": [{
                "equation": {
                    "content": "u(\\mathbf{x},t) = \\sum_{n=1}^{\\infty} A_n e^{-\\alpha \\lambda_n t} \\phi_n(\\mathbf{x})"
                }
            }]
        }
    },
]

client.api(
    "POST",
    f"/docx/v1/documents/{document_id}/blocks/{document_id}/children",
    json_data={"children": pde_blocks}
)
print("[OK] Appended PDE formulas")

[OK] Appended PDE formulas


### 4.3 数论公式

黎曼 ζ 函数的函数方程与质数定理。

In [41]:
number_theory_blocks = [
    make_heading_block("数论核心公式", level=3),
    make_text_block("黎曼 ζ 函数的函数方程："),
    {
        "block_type": 2,
        "text": {
            "elements": [{
                "equation": {
                    "content": "\\zeta(s) = 2^{s} \\pi^{s-1} \\sin\\left( \\frac{\\pi s}{2} \\right) \\Gamma(1-s) \\zeta(1-s)"
                }
            }]
        }
    },
    make_text_block("质数定理（素数计数函数）："),
    {
        "block_type": 2,
        "text": {
            "elements": [{
                "equation": {
                    "content": "\\pi(x) \\sim \\frac{x}{\\ln x} \\sim \\text{Li}(x) = \\int_{2}^{x} \\frac{dt}{\\ln t}"
                }
            }]
        }
    },
    make_text_block("欧拉乘积公式："),
    {
        "block_type": 2,
        "text": {
            "elements": [{
                "equation": {
                    "content": "\\zeta(s) = \\sum_{n=1}^{\\infty} \\frac{1}{n^{s}} = \\prod_{p \\text{ prime}} \\frac{1}{1 - p^{-s}}, \\quad \\text{Re}(s) > 1"
                }
            }]
        }
    },
]

client.api(
    "POST",
    f"/docx/v1/documents/{document_id}/blocks/{document_id}/children",
    json_data={"children": number_theory_blocks}
)
print("[OK] Appended number theory formulas")

[OK] Appended number theory formulas


### 4.4 Scaled Dot-Product Attention

Attention Is All You Need 中的核心注意力计算。

In [42]:
attention_blocks = [
    make_heading_block("Scaled Dot-Product Attention", level=3),
    make_text_block("核心公式："),
    {
        "block_type": 2,
        "text": {
            "elements": [{
                "equation": {
                    "content": "\\text{Attention}(Q, K, V) = \\text{softmax}\\left( \\frac{QK^{T}}{\\sqrt{d_k}} \\right) V"
                }
            }]
        }
    },
    make_text_block("Mask 版本（因果/自回归）："),
    {
        "block_type": 2,
        "text": {
            "elements": [{
                "equation": {
                    "content": "\\text{Attention}(Q, K, V) = \\text{softmax}\\left( \\frac{QK^{T}}{\\sqrt{d_k}} + M \\right) V, \\quad M_{ij} = \\begin{cases} 0 & i \\geq j \\\\ -\\infty & i < j \\end{cases}"
                }
            }]
        }
    },
]

client.api(
    "POST",
    f"/docx/v1/documents/{document_id}/blocks/{document_id}/children",
    json_data={"children": attention_blocks}
)
print("[OK] Appended Attention formula")

[OK] Appended Attention formula


### 4.5 Multi-Head Attention 完整推导

多头注意力的线性投影、并行计算与输出拼接。

In [43]:
mha_blocks = [
    make_heading_block("Multi-Head Attention", level=3),
    make_text_block("第 i 个注意力头的线性投影："),
    {
        "block_type": 2,
        "text": {
            "elements": [{
                "equation": {
                    "content": "Q_i = X W_i^{Q}, \\quad K_i = X W_i^{K}, \\quad V_i = X W_i^{V}"
                }
            }]
        }
    },
    make_text_block("单头注意力输出："),
    {
        "block_type": 2,
        "text": {
            "elements": [{
                "equation": {
                    "content": "\\text{head}_i = \\text{Attention}(Q_i, K_i, V_i) = \\text{softmax}\\left( \\frac{Q_i K_i^{T}}{\\sqrt{d_k}} \\right) V_i"
                }
            }]
        }
    },
    make_text_block("多头拼接与输出投影："),
    {
        "block_type": 2,
        "text": {
            "elements": [{
                "equation": {
                    "content": "\\text{MultiHead}(Q, K, V) = \\text{Concat}(\\text{head}_1, \\ldots, \\text{head}_h) W^{O}"
                }
            }]
        }
    },
    make_text_block("维度关系（d_model = h × d_k）："),
    {
        "block_type": 2,
        "text": {
            "elements": [{
                "equation": {
                    "content": "W_i^{Q}, W_i^{K}, W_i^{V} \\in \\mathbb{R}^{d_{model} \\times d_k}, \\quad W^{O} \\in \\mathbb{R}^{hd_k \\times d_{model}}"
                }
            }]
        }
    },
]

client.api(
    "POST",
    f"/docx/v1/documents/{document_id}/blocks/{document_id}/children",
    json_data={"children": mha_blocks}
)
print("[OK] Appended MHA formulas")

[OK] Appended MHA formulas


### 4.6 MoE Auxiliary Loss

Mixture of Experts 的路由均衡损失，防止所有 token 都路由到同一个专家。

In [44]:
moe_blocks = [
    make_heading_block("MoE Auxiliary Loss", level=3),
    make_text_block("专家 i 的 token 比例（实际路由到的比例）："),
    {
        "block_type": 2,
        "text": {
            "elements": [{
                "equation": {
                    "content": "f_i = \\frac{1}{T} \\sum_{x \\in \\mathcal{B}} \\mathbb{1}\\{ \\text{argmax}_j \\, p_j(x) = i \\}"
                }
            }]
        }
    },
    make_text_block("专家 i 的平均路由概率（路由器分配的期望）："),
    {
        "block_type": 2,
        "text": {
            "elements": [{
                "equation": {
                    "content": "P_i = \\frac{1}{T} \\sum_{x \\in \\mathcal{B}} p_i(x)"
                }
            }]
        }
    },
    make_text_block("Auxiliary Loss（负载均衡惩罚）："),
    {
        "block_type": 2,
        "text": {
            "elements": [{
                "equation": {
                    "content": "\\mathcal{L}_{aux} = \\alpha \\cdot N \\cdot \\sum_{i=1}^{N} f_i \\cdot P_i"
                }
            }]
        }
    },
    make_text_block("总损失（主损失 + 辅助损失）："),
    {
        "block_type": 2,
        "text": {
            "elements": [{
                "equation": {
                    "content": "\\mathcal{L}_{total} = \\mathcal{L}_{main} + \\mathcal{L}_{aux}"
                }
            }]
        }
    },
]

client.api(
    "POST",
    f"/docx/v1/documents/{document_id}/blocks/{document_id}/children",
    json_data={"children": moe_blocks}
)
print("[OK] Appended MoE AuxLoss formulas")

[OK] Appended MoE AuxLoss formulas


## 5. 追加表格

直接构造 `block_type: 31` table + `block_type: 32` table_cell 块。

In [45]:
# 追加表格（飞书 table 需要分步创建：空 table → GET cells → PATCH 每个 cell）
# Step 1: 创建空 table（只含 property）
empty_table = {
    "block_type": 31,
    "table": {
        "property": {
            "row_size": 3,
            "column_size": 3,
            "merge_type": 0,
            "header_row": True,
            "header_column": False
        }
    }
}

result = client.api(
    "POST",
    f"/docx/v1/documents/{document_id}/blocks/{document_id}/children",
    json_data={"children": [make_heading_block("表格测试", level=2), empty_table]}
)

# 注意：children[0] 是 heading2，children[1] 才是 table
table_block_result = [c for c in result["children"] if c.get("block_type") == 31][0]
table_id = table_block_result["block_id"]
cell_ids = table_block_result["table"]["cells"]
print(f"[OK] Created empty table: {table_id}")
print(f"     Cells: {len(cell_ids)}")

# Step 2: 准备每个 cell 的内容（按行优先顺序）
cell_contents = [
    [{"text_run": {"content": "姓名", "text_element_style": {}}}],
    [{"text_run": {"content": "年龄", "text_element_style": {}}}],
    [{"text_run": {"content": "城市", "text_element_style": {}}}],
    [{"text_run": {"content": "Alice", "text_element_style": {}}}],
    [{"text_run": {"content": "25", "text_element_style": {}}}],
    [{"text_run": {"content": "北京", "text_element_style": {}}}],
    [{"text_run": {"content": "Bob", "text_element_style": {}}}],
    [{"text_run": {"content": "30", "text_element_style": {}}}],
    [{"text_run": {"content": "上海", "text_element_style": {}}}],
]

# Step 3: 对每个 cell，GET 获取 auto-generated text child → PATCH 更新
import time
for idx, cell_id in enumerate(cell_ids):
    # 3a. GET cell 获取 auto-generated text child
    cell_result = client.api("GET", f"/docx/v1/documents/{document_id}/blocks/{cell_id}")
    text_child_id = cell_result.get("block", {}).get("children", [None])[0]
    if not text_child_id:
        print(f"[WARN] Cell {idx}: no auto-generated text child")
        continue
    
    # 3b. PATCH text child 内容
    patch_result = client.api(
        "PATCH",
        f"/docx/v1/documents/{document_id}/blocks/{text_child_id}",
        json_data={"update_text_elements": {"elements": cell_contents[idx]}}
    )
    # PATCH 成功时返回 block 对象，没有 code 字段
    if patch_result.get("code", 0) == 0 or "block" in patch_result:
        print(f"[OK] Cell {idx} updated")
    else:
        print(f"[WARN] Cell {idx} patch failed: {patch_result.get('msg')}")
    
    # QPS 保护：每 3 个 cell 延时 400ms
    if idx > 0 and idx % 3 == 0:
        time.sleep(0.4)

print("[OK] Table fully created and populated")

[OK] Created empty table: doxcnPrUx4Coez9RLdk87TELKnc
     Cells: 9
[OK] Cell 0 updated
[OK] Cell 1 updated
[OK] Cell 2 updated
[OK] Cell 3 updated
[OK] Cell 4 updated
[OK] Cell 5 updated
[OK] Cell 6 updated
[OK] Cell 7 updated
[OK] Cell 8 updated
[OK] Table fully created and populated


## 6. 准备测试图片


In [46]:
from PIL import Image, ImageDraw, ImageFont
import io

img = Image.new('RGB', (400, 200), color=(73, 109, 137))
draw = ImageDraw.Draw(img)
try:
    font = ImageFont.truetype("arial.ttf", 24)
except:
    font = ImageFont.load_default()
draw.text((20, 80), "Test Image for Feishu", fill=(255, 255, 255), font=font)

buf = io.BytesIO()
img.save(buf, format='PNG')
img_bytes = buf.getvalue()

print(f"[OK] Test image created: {len(img_bytes)} bytes")

[OK] Test image created: 4195 bytes


## 7. 插入图片（三步法）

飞书 docx 插入图片必须走 **三步法**，直接带 token 创建块会报 `1770001 invalid param`：

1. **创建空图片块** → 拿到 `image_block_id`
2. **上传图片素材** → `parent_node` 必须填**图片块ID**，`parent_type="docx_image"`
3. **PATCH 绑定** → 用 `replace_image` 把素材 token 挂到块上

> ⚠️ 注意：`parent_node` 必须是图片块的 `block_id`，**绝对不能**填 `document_id`，否则报 `relation mismatch`。

In [47]:
import time

# Step 1: 创建空图片块（只传空对象，不填 token）
empty_image_block = {"block_type": 27, "image": {}}
img_result = client.api(
    "POST",
    f"/docx/v1/documents/{document_id}/blocks/{document_id}/children",
    json_data={"children": [make_heading_block("图片测试", level=2), empty_image_block]}
)

# 从返回结果里找到刚创建的图片块
image_block_result = [c for c in img_result["children"] if c.get("block_type") == 27][0]
image_block_id = image_block_result["block_id"]
print(f"[OK] Created empty image block: {image_block_id}")

time.sleep(0.5)

# Step 2: 上传图片素材，parent_node 必须是图片块 ID！
upload_result = client.request(
    "POST",
    "/drive/v1/medias/upload_all",
    files={"file": ("test.png", img_bytes, "image/png")},
    data={
        "file_name": "test.png",
        "parent_type": "docx_image",   # docx 图片必须用 docx_image
        "parent_node": image_block_id,  # 必须是图片块 ID，不能是 document_id！
        "size": str(len(img_bytes)),
    }
)

if upload_result.get("code", 0) == 0 and "file_token" in upload_result.get("data", {}):
    file_token = upload_result["data"]["file_token"]
    print(f"[OK] Image uploaded, file_token: {file_token}")
    
    # Step 3: PATCH 绑定图片到块
    client.api(
        "PATCH",
        f"/docx/v1/documents/{document_id}/blocks/{image_block_id}",
        json_data={"replace_image": {"token": file_token}}
    )
    print("[OK] Image bound to block successfully")
else:
    err_msg = upload_result.get('msg', str(upload_result)[:200])
    print(f"[WARN] Upload failed: {err_msg}")
    file_token = None

[OK] Created empty image block: doxcnaZomykc2Qwc5Brx7SvBGWb
[OK] Image uploaded, file_token: DT3WbbUoVoja6Tx9mkNcuKwInVc
[OK] Image bound to block successfully


## 7.5 插入网络图片

从网上下载图片并插入文档，三步法和本地图片完全一致，区别只是 `img_bytes` 的来源。

In [48]:
import requests
import time
from urllib.parse import urlparse

# ================== 填你要的图片 URL ==================
IMAGE_URL = "https://picsum.photos/800/400"   # 随机风景图（可换任意图片链接）
# 其他测试图：
# IMAGE_URL = "https://www.google.com/images/branding/googlelogo/2x/googlelogo_color_92x30dp.png"
# IMAGE_URL = "https://upload.wikimedia.org/wikipedia/commons/thumb/1/1a/Transformer_architecture.png/640px-Transformer_architecture.png"
# ======================================================

# Step 0: 从网上下载图片
print(f"--- 正在下载图片: {IMAGE_URL} ---")
img_resp = requests.get(IMAGE_URL, timeout=30)
img_resp.raise_for_status()
img_bytes = img_resp.content
print(f"✅ 下载完成: {len(img_bytes)} bytes")

# 文件名从 URL 提取
parsed = urlparse(IMAGE_URL)
file_name = parsed.path.split("/")[-1] or "downloaded_image.png"
if "." not in file_name:
    file_name += ".png"

# Step 1: 创建空图片块
empty_image_block = {"block_type": 27, "image": {}}
img_result = client.api(
    "POST",
    f"/docx/v1/documents/{document_id}/blocks/{document_id}/children",
    json_data={"children": [make_heading_block("网络图片测试", level=2), empty_image_block]}
)

image_block_result = [c for c in img_result["children"] if c.get("block_type") == 27][0]
image_block_id = image_block_result["block_id"]
print(f"[OK] Created empty image block: {image_block_id}")

time.sleep(0.5)

# Step 2: 上传图片素材
upload_result = client.request(
    "POST",
    "/drive/v1/medias/upload_all",
    files={"file": (file_name, img_bytes, img_resp.headers.get("content-type", "image/png"))},
    data={
        "file_name": file_name,
        "parent_type": "docx_image",
        "parent_node": image_block_id,
        "size": str(len(img_bytes)),
    }
)

if upload_result.get("code", 0) == 0 and "file_token" in upload_result.get("data", {}):
    file_token = upload_result["data"]["file_token"]
    print(f"[OK] Image uploaded, file_token: {file_token}")
    
    # Step 3: PATCH 绑定
    client.api(
        "PATCH",
        f"/docx/v1/documents/{document_id}/blocks/{image_block_id}",
        json_data={"replace_image": {"token": file_token}}
    )
    print("[OK] Image bound to block successfully")
else:
    err_msg = upload_result.get('msg', str(upload_result)[:200])
    print(f"[WARN] Upload failed: {err_msg}")

--- 正在下载图片: https://picsum.photos/800/400 ---
✅ 下载完成: 58609 bytes
[OK] Created empty image block: doxcnOA3z3Vtq6hCEwVSAALeh9e
[OK] Image uploaded, file_token: AEwWbw2dwonp79xqA4ScsMxqnYb
[OK] Image bound to block successfully


## 8. 读取文档

验证 `GET /docx/v1/documents/{id}/content` 读取纯文本。

In [49]:
read_result = client.api("GET", f"/docx/v1/documents/{document_id}/raw_content")
raw_text = read_result.get("content", "")
print(f"[OK] Read doc, content length: {len(raw_text)} chars")
lines = [l.strip() for l in raw_text.split('\n') if l.strip()]
for line in lines[:10]:
    print(f"  {line[:60]}")

[OK] Read doc, content length: 712 chars
  API验证 - 01:22:55
  文本追加测试
  这是一段普通文本，用于验证 feishu_doc_append 的 content 模式。
  支持无序列表
  自动转换为飞书 bullet block
  有序列表项1
  有序列表项2
  引用块测试
  段落之间需要空行分隔。
  代码块测试


## 9. 获取块结构

验证 `GET /docx/v1/documents/{id}/blocks` 获取文档块列表，用于后续更新/删除。

In [50]:
blocks_result = client.api("GET", f"/docx/v1/documents/{document_id}/blocks", params={"page_size": 500})
block_items = blocks_result.get("items", [])

print(f"[OK] Got {len(block_items)} blocks")
print(f"{'Block ID':<30} {'Type':<15} {'Preview'}")
print("-" * 70)
for b in block_items[:15]:
    bt = b.get("block_type", 0)
    bid = b.get("block_id", "")[:28]
    preview = extract_text_from_block(b)[:30]
    print(f"{bid:<30} {block_type_name(bt):<15} {preview}")

[OK] Got 83 blocks
Block ID                       Type            Preview
----------------------------------------------------------------------
HgGBdPSVqocd5UxYCoocxh7cnYc    page            
doxcngRGsTIqJWh1G7wlEowCoEy    heading2        文本追加测试
doxcnUx1DIUvsoUwmq5ElABQeeh    text            这是一段普通文本，用于验证 feishu_doc_appen
doxcn8GYaosu9GCddQF62FBOSZc    bullet          支持无序列表
doxcnhD9ZwULqVwkWnyZE1rZOAW    bullet          自动转换为飞书 bullet block
doxcnJuSEBArwCyrDHDFgdy2Ysd    ordered         有序列表项1
doxcn9txcce8yjTRwOrwnUBEO7c    ordered         有序列表项2
doxcnR8Klo8il2BEc9sG68dSXIb    quote           引用块测试
doxcn7GRMcJIMAFcT3YL2HUp3zd    text            段落之间需要空行分隔。
doxcnVuyVlgucKtsydYKkV707qd    heading2        代码块测试
doxcnjLRc5HjZN38AtPulLKl4sg    text            Python 示例代码:
doxcnxKMKrJRSVBzFUxxO0m3ROe    code            def hello():
    print('Hello,
doxcnFXuwqdVgyDlPIZJxB9VToe    heading2        数学公式测试
doxcnyiSu7jAZkwWTyiofV82PRd    text            一元二次方程求根公式：
doxcnAgtrFxqior87oGCeAEnHnb  

## 10. 更新指定块

验证 `PUT /docx/v1/documents/{id}/blocks/{block_id}` 更新块内容。

In [51]:
# 更新第一个文本块，让最终文档中一眼能看出"这是更新后的内容"
text_blocks = [b for b in block_items if b.get("block_type") == 2]
if text_blocks:
    target = text_blocks[0]
    target_id = target["block_id"]
    client.api(
        "PUT",
        f"/docx/v1/documents/{document_id}/blocks/{target_id}",
        json_data={
            "replace_block": {
                "block_type": 2,
                "text": {
                    "elements": [{
                        "text_run": {
                            "content": "【更新后】这段文本已被 feishu_doc_update_block 修改，如果你看到这句话，说明更新成功",
                            "text_element_style": {}
                        }
                    }]
                }
            }
        }
    )
    print(f"[OK] Updated block {target_id}")
else:
    print("[SKIP] No text block found")

[OK] Updated block doxcnUx1DIUvsoUwmq5ElABQeeh


## 11. 搜索文档

验证 `POST /suite/docs-api/search/object` 搜索云空间文档。

In [52]:
try:
    search_result = client.api(
        "POST",
        "/suite/docs-api/search/object",
        json_data={"search_key": "API", "count": 5}
    )
    docs = search_result.get("docs_entities", [])
    print(f"[OK] Found {len(docs)} docs")
    for d in docs[:3]:
        print(f"  - {d.get('title', 'N/A')} ({d.get('type', 'N/A')})")
except Exception as e:
    print(f"[WARN] Search failed: {e}")

[OK] Found 5 docs
  - API 参考 (N/A)
  - API验证 - 01:21:11 (N/A)
  - API验证 - 01:12:57 (N/A)


## 12. 搜索用户

验证 `POST /contact/v3/users/batch_get_id` 用户查找。

**注意**：需要应用有通讯录权限。

In [53]:
try:
    user_result = client.api(
        "POST",
        "/contact/v3/users/batch_get_id",
        json_data={"emails": ["test@example.com"]},
        params={"user_id_type": "open_id"}
    )
    users = user_result.get("user_list", [])
    print(f"[OK] Found {len(users)} users")
    for u in users:
        print(f"  - {u.get('user_id', 'N/A')}")
except Exception as e:
    print(f"[WARN] User search failed: {e}")

[OK] Found 1 users
  - N/A


## 13. 删除测试块

验证 `DELETE /docx/v1/documents/{id}/blocks/{block_id}` 删除指定块。

In [54]:
import time

# 13-1. 先添加标记块
delete_marker = make_text_block("【66666下面的内容应该被删除。如果最终文档中看不到这句话，说明删除成功】")
add_result = client.api(
    "POST",
    f"/docx/v1/documents/{document_id}/blocks/{document_id}/children",
    json_data={"children": [delete_marker]}
)
marker_id = add_result["children"][0]["block_id"]
print(f"[OK] Added marker block: {marker_id}")

[OK] Added marker block: doxcnPrISpFsXaZueezW1ZkIAJg


In [55]:
# 13-2 等待块更新到根节点 children 列表（有时可能有短暂延迟），再进行删除
time.sleep(0.5)

# 2. 获取根块的 children 列表，找到标记块的索引
root_block = client.api("GET", f"/docx/v1/documents/{document_id}/blocks/{document_id}")
children = root_block.get("block", {}).get("children", [])

if marker_id in children:
    index = children.index(marker_id)
    print(f"[INFO] Marker index in root.children: {index}")

    # 3. 用 batch_delete 从根节点删除这个子块
    client.api(
        "DELETE",
        f"/docx/v1/documents/{document_id}/blocks/{document_id}/children/batch_delete",
        json_data={"start_index": index, "end_index": index + 1}
    )
    print(f"[OK] Deleted marker block via batch_delete")
else:
    print(f"[WARN] Marker block not found in root children")

[INFO] Marker index in root.children: 64
[OK] Deleted marker block via batch_delete


## 15. 知识库（Wiki）操作

飞书知识库（Wiki）用于组织和管理文档的层级结构。


In [56]:
# 15.1 创建知识库空间（保留）
import time
wiki_name = f'API测试知识库 {time.strftime("%H:%M:%S")}'
wiki_desc = '由 feishu_client 自动创建的知识库，用于 API 验证'
wiki_result = client.create_wiki_space(name=wiki_name, description=wiki_desc)
WIKI_SPACE_ID = wiki_result['space']['space_id']
print(f'创建知识库: {WIKI_SPACE_ID}')
print(json.dumps(wiki_result, indent=2, ensure_ascii=False))

创建知识库: 7632377606416436165
{
  "space": {
    "description": "由 feishu_client 自动创建的知识库，用于 API 验证",
    "name": "API测试知识库 01:23:14",
    "space_id": "7632377606416436165",
    "space_type": "team",
    "visibility": "private"
  }
}


In [57]:
time.sleep(5)  # 等待知识库创建完成
# 15.2 列出知识库空间
spaces = client.list_wiki_spaces(page_size=10)
print(f'知识库数量: {len(spaces)}')
for sp in spaces:        # ← 去掉 [:5]
    print(f"  - {sp.get('name')} (ID: {sp.get('space_id')})")

知识库数量: 8
  - API测试知识库 01:13:18 (ID: 7632375046165531611)
  - 应知序资料库 (ID: 7370571654834995201)
  - 🚀面壁小钢炮MiniCPM (ID: 7450911071910543363)
  - 赋范空间-大模型技术社区 (ID: 7450744448952074268)
  - forkLLM知识库 (ID: 7535890214334595100)
  - LLM从入门到入土 (ID: 7535467103394988036)
  - API测试知识库 01:21:32 (ID: 7632377167329938372)
  - API测试知识库 01:23:14 (ID: 7632377606416436165)


In [58]:
# 15.3 获取知识库详情
space_detail = client.get_wiki_space(WIKI_SPACE_ID)
print(json.dumps(space_detail, indent=2, ensure_ascii=False))

{
  "space": {
    "description": "由 feishu_client 自动创建的知识库，用于 API 验证",
    "name": "API测试知识库 01:23:14",
    "open_sharing": "closed",
    "space_id": "7632377606416436165",
    "space_type": "team",
    "visibility": "private"
  }
}


In [59]:
# 15.4 更新知识库信息
updated = client.update_wiki_space(
    WIKI_SPACE_ID,
    name=f'{wiki_name} [已更新]',
    description='更新后的描述'
)
print('更新结果:', json.dumps(updated, indent=2, ensure_ascii=False))

# 查看更新后的知识库
print('--- 更新后 ---')
space_detail = client.get_wiki_space(WIKI_SPACE_ID)
print(json.dumps(space_detail, indent=2, ensure_ascii=False))

更新结果: {
  "raw_text": "404 page not found",
  "status_code": 404
}
--- 更新后 ---
{
  "space": {
    "description": "由 feishu_client 自动创建的知识库，用于 API 验证",
    "name": "API测试知识库 01:23:14",
    "open_sharing": "closed",
    "space_id": "7632377606416436165",
    "space_type": "team",
    "visibility": "private"
  }
}


In [60]:
# 15.5 在知识库中创建两个文档节点
node_1 = client.create_wiki_node(
    space_id=WIKI_SPACE_ID,
    node_type='origin',
    obj_type='docx',
    title='文档1-保留'
)
WIKI_NODE_TOKEN_1 = node_1['node']['node_token']
NODE_DOC_ID_1 = node_1['node']['obj_token']
print(f'文档1: token={WIKI_NODE_TOKEN_1}, doc_id={NODE_DOC_ID_1}')

node_2 = client.create_wiki_node(
    space_id=WIKI_SPACE_ID,
    node_type='origin',
    obj_type='docx',
    title='文档2-待删'
)
WIKI_NODE_TOKEN_2 = node_2['node']['node_token']
NODE_DOC_ID_2 = node_2['node']['obj_token']
print(f'文档2: token={WIKI_NODE_TOKEN_2}, doc_id={NODE_DOC_ID_2}')

文档1: token=IUq5w5vnDiMXrFkBbgqcGQWKnIh, doc_id=JniSdXbYXoB8FZxRxdtc6S2KnBb
文档2: token=Xr2ZwzG4eidY71k3wDVcCfm1nob, doc_id=Z1Tjd0la8oy8hrxNj7PcLhQ5nVc


In [61]:
# 15.5b 在文档1下创建子文档
child = client.create_wiki_node(
    space_id=WIKI_SPACE_ID,
    node_type='origin',
    obj_type='docx',
    parent_node_token=WIKI_NODE_TOKEN_1,  # 挂在文档1下面
    title='子文档'
)
CHILD_NODE_TOKEN = child['node']['node_token']
CHILD_DOC_ID = child['node']['obj_token']
print(f'子文档: token={CHILD_NODE_TOKEN}, doc_id={CHILD_DOC_ID}')

子文档: token=RUm9wVnAoiuftakkQpZc7VtYnCg, doc_id=WK1DdnVNQorDG0xqZoycAmpZn1f


### 15.5c 外部文档完整生命周期：创建 → 写入 → 迁入 Wiki → 修改

验证外部 docx 在迁入知识库前后的内容读写能力。

In [62]:
# Step 1: 创建外部文档（必须用 user_access_token，否则后续无权限迁入 Wiki）
external_title = f'外部文档-{time.strftime("%H%M%S")}'
external_doc = client.api('POST', '/docx/v1/documents', json_data={'title': external_title}, use_user_token=True)
EXTERNAL_DOC_ID = external_doc['document']['document_id']
print(f'[1/4] 创建外部文档: {EXTERNAL_DOC_ID}')

# Step 2: 写入初始内容
init_blocks = [
    make_heading_block('初始内容', level=2),
    make_text_block('这是外部文档的初始内容，在迁入 Wiki 之前写入。'),
    make_text_block(f'创建时间: {time.strftime("%Y-%m-%d %H:%M:%S")}'),
]
client.api(
    'POST',
    f'/docx/v1/documents/{EXTERNAL_DOC_ID}/blocks/{EXTERNAL_DOC_ID}/children',
    json_data={'children': init_blocks},
    use_user_token=True
)
print('[2/4] 初始内容写入完成')

# Step 3: 迁入 Wiki 知识库
moved = client.move_doc_to_wiki(
    space_id=WIKI_SPACE_ID,
    doc_token=EXTERNAL_DOC_ID
)
EXTERNAL_NODE_TOKEN = moved.get('node', {}).get('node_token', 'N/A')
print(f'[3/4] 迁入 Wiki 完成, node_token={EXTERNAL_NODE_TOKEN}')

# Step 4: 迁入后修改内容
time.sleep(1)  # 等待索引同步
update_blocks = [
    make_heading_block('迁入后追加', level=2),
    make_text_block('这篇文档已从外部迁入 Wiki 知识库，这是迁入后追加的内容。'),
    make_text_block('验证成功：外部文档迁入 Wiki 后仍可正常读写。'),
]
client.api(
    'POST',
    f'/docx/v1/documents/{EXTERNAL_DOC_ID}/blocks/{EXTERNAL_DOC_ID}/children',
    json_data={'children': update_blocks},
    use_user_token=True
)
print('[4/4] 迁入后内容修改完成')

# 查看最终内容
final_content = client.api('GET', f'/docx/v1/documents/{EXTERNAL_DOC_ID}/raw_content', use_user_token=True)
print('\n--- 文档最终内容 ---')
print(final_content.get('content', '无内容')[:400])

[1/4] 创建外部文档: KSrpdX5Nzoyk0yxfeBNcqmQcnTf
[2/4] 初始内容写入完成
[3/4] 迁入 Wiki 完成, node_token=N/A
[4/4] 迁入后内容修改完成

--- 文档最终内容 ---
外部文档-012325
初始内容
这是外部文档的初始内容，在迁入 Wiki 之前写入。
创建时间: 2026-04-25 01:23:26
迁入后追加
这篇文档已从外部迁入 Wiki 知识库，这是迁入后追加的内容。
验证成功：外部文档迁入 Wiki 后仍可正常读写。



In [63]:
# 15.6 列出知识库节点
nodes = client.list_wiki_nodes(WIKI_SPACE_ID, page_size=10)
print(f'节点数量: {len(nodes)}')
for n in nodes:
    print(f"  - {n.get('title')} ({n.get('obj_type')}, token={n.get('node_token')})")

节点数量: 3
  - 文档1-保留 (docx, token=IUq5w5vnDiMXrFkBbgqcGQWKnIh)
  - 文档2-待删 (docx, token=Xr2ZwzG4eidY71k3wDVcCfm1nob)
  - 外部文档-012325 (docx, token=Wh7jwYcicirPDTksxzncG61inbe)


In [64]:
# 15.7 获取文档1结构、追加内容、并查看

# 1. 获取文档块结构
blocks_result = client.api('GET', f'/docx/v1/documents/{NODE_DOC_ID_1}/blocks', use_user_token=True)
block_items = blocks_result.get('items', [])
print(f'文档1块数量: {len(block_items)}')

# 2. 找到 page block（block_type=1）
page_block = next((b for b in block_items if b.get('block_type') == 1), None)
if not page_block:
    print('未找到 page block')
else:
    page_id = page_block['block_id']
    
    # 3. 在 page block 下追加内容
    client.api(
        'POST',
        f'/docx/v1/documents/{NODE_DOC_ID_1}/blocks/{page_id}/children',
        json_data={
            'children': [
                make_text_block('这是文档1的内容，跑完 notebook 后可在飞书知识库中查看。')
            ]
        },
        use_user_token=True
    )
    print('文档1内容追加成功')
    
    # 4. 查看文档内容
    print('--- 文档1当前内容 ---')
    content = client.api('GET', f'/docx/v1/documents/{NODE_DOC_ID_1}/raw_content', use_user_token=True)
    print(content.get('content', '无内容')[:300])

文档1块数量: 1
文档1内容追加成功
--- 文档1当前内容 ---
文档1-保留
这是文档1的内容，跑完 notebook 后可在飞书知识库中查看。



In [65]:
# 15.8 删除 Wiki 节点 —— 飞书未提供 API，请手动在客户端删除
print('【提示】飞书开放平台未提供 Wiki 节点删除 API')
print(f'请手动删除: 文档2-待删 (token={WIKI_NODE_TOKEN_2})')
print('操作路径: 飞书 → 知识库 → 找到文档 → 右键 → 删除')

# 查看当前节点
print('--- 当前节点列表 ---')
nodes = client.list_wiki_nodes(WIKI_SPACE_ID, page_size=10)
print(f'节点数量: {len(nodes)}')
for n in nodes:
    print(f"  - {n.get('title')} ({n.get('obj_type')}, token={n.get('node_token')})")

【提示】飞书开放平台未提供 Wiki 节点删除 API
请手动删除: 文档2-待删 (token=Xr2ZwzG4eidY71k3wDVcCfm1nob)
操作路径: 飞书 → 知识库 → 找到文档 → 右键 → 删除
--- 当前节点列表 ---
节点数量: 3
  - 文档1-保留 (docx, token=IUq5w5vnDiMXrFkBbgqcGQWKnIh)
  - 文档2-待删 (docx, token=Xr2ZwzG4eidY71k3wDVcCfm1nob)
  - 外部文档-012325 (docx, token=Wh7jwYcicirPDTksxzncG61inbe)


In [66]:
# 15.9 删除知识库 —— 不想删就注释掉下面两行
# result = client.delete_wiki_space(WIKI_SPACE_ID)
# print(f'删除结果: {json.dumps(result, indent=2, ensure_ascii=False)}')

# 查看剩余知识库
print('--- 删除后知识库列表 ---')
spaces = client.list_wiki_spaces(page_size=10)
print(f'剩余知识库数量: {len(spaces)}')
for sp in spaces:
    print(f"  - {sp.get('name')} (ID: {sp.get('space_id')})")

--- 删除后知识库列表 ---
剩余知识库数量: 8
  - API测试知识库 01:13:18 (ID: 7632375046165531611)
  - 应知序资料库 (ID: 7370571654834995201)
  - 🚀面壁小钢炮MiniCPM (ID: 7450911071910543363)
  - 赋范空间-大模型技术社区 (ID: 7450744448952074268)
  - forkLLM知识库 (ID: 7535890214334595100)
  - LLM从入门到入土 (ID: 7535467103394988036)
  - API测试知识库 01:21:32 (ID: 7632377167329938372)
  - API测试知识库 01:23:14 (ID: 7632377606416436165)


## 16. 验证总结

| 功能 | 状态 |
|------|------|
| 创建文档 | ✅ |
| 追加文本 | ✅ |
| 追加代码块 | ✅ |
| 追加数学公式 | ✅ |
| 追加表格 | ✅ |
| 上传图片 | ✅ |
| 读取文档 | ✅ |
| 获取块结构 | ✅ |
| 更新块 | ✅ |
| 搜索文档 | ✅ |
| 搜索用户 | ✅ |
| 删除块 | ✅ |
| 知识库创建/读取/更新/删除 | ✅ |
| 知识库节点挂载/删除 | ✅ |
| 外部文档创建→写入→迁入→修改 | ✅ |


In [67]:
print('=' * 60)
print('飞书 API 客户端验证全部通过！')
print('=' * 60)


飞书 API 客户端验证全部通过！
